# Merge IRGA75 (no self-heating correction) with IRGA72 data (2016-2017)
- NEE is prepared for the correction from parallel measurements.
- Note that the NEE used here is not really NEE, here NEE is the same as FC because FC is corrected for self-heating and not NEE
- LE was not corrected because IRGA75 LE was consistently higher than IRGA72 LE, but the self-heating correction can only *ADD* flux, not reduce it.

In [1]:
from datetime import datetime
import pandas as pd
from diive.core.io.files import save_parquet, load_parquet

# Required variables

In [2]:
# Variable names in EddyPro _fluxnet_ output files
AIR_CP = "AIR_CP"
AIR_DENSITY = "AIR_DENSITY"
VAPOR_DENSITY = "VAPOR_DENSITY"
U = "U"
USTAR = "USTAR"
TA = "TA_T1_47_1_gfXG"
RH = "RH_T1_47_1"
SWIN = "SW_IN_T1_47_1_gfXG"
CO2_MOLAR_DENSITY = "CO2_MOLAR_DENSITY"
NEE = "NEE_L3.1_L3.2_QCF"

# Load IRGA75 data

In [3]:
df_irga75 = load_parquet(filepath="02_IRGA75_FPC_NEE-QCF11-L3.2_2016-2017.parquet")
df_irga75 = df_irga75[[NEE, CO2_MOLAR_DENSITY]].copy()
df_irga75 = df_irga75.add_suffix('_IRGA75')
df_irga75.describe()
# [print(c) for c in df_nee_irga75 if "RH" in c];

Loaded .parquet file 02_IRGA75_FPC_NEE-QCF11-L3.2_2016-2017.parquet (0.092 seconds).
    --> Detected time resolution of <30 * Minutes> / 30min 


,NEE_L3.1_L3.2_QCF_IRGA75,CO2_MOLAR_DENSITY_IRGA75
count,17617.000000,31180.000000
mean,-5.472220,16.537655
std,10.030694,2.240562
min,-49.996500,8.497130
25%,-10.601500,15.344950
50%,-2.067410,16.026400
75%,0.768966,16.810625
max,30.589900,34.866200


# Load IRGA72 data

In [4]:
df_irga72 = load_parquet(filepath="12_IRGA72_FPC_NEE-QCF11-L3.2_2016-2017.parquet")
df_irga72 = df_irga72[[NEE, CO2_MOLAR_DENSITY, AIR_CP, AIR_DENSITY, VAPOR_DENSITY, U, USTAR, TA, SWIN, RH]].copy()
df_irga72 = df_irga72.add_suffix('_IRGA72')
df_irga72.describe()

Loaded .parquet file 12_IRGA72_FPC_NEE-QCF11-L3.2_2016-2017.parquet (0.057 seconds).
    --> Detected time resolution of <30 * Minutes> / 30min 


,NEE_L3.1_L3.2_QCF_IRGA72,CO2_MOLAR_DENSITY_IRGA72,AIR_CP_IRGA72,AIR_DENSITY_IRGA72,VAPOR_DENSITY_IRGA72,U_IRGA72,USTAR_IRGA72,TA_T1_47_1_gfXG_IRGA72,SW_IN_T1_47_1_gfXG_IRGA72,RH_T1_47_1_IRGA72
count,17402.000000,27474.000000,27749.000000,27749.000000,27749.000000,27749.000000,27749.000000,28032.000000,28032.000000,28028.000000
mean,-2.866511,15.574433,1011.736249,1.141921,0.008234,2.447515,0.465091,10.725810,155.227218,80.873202
std,9.000624,1.154616,2.845976,0.037616,0.003242,1.715387,0.296126,8.527703,250.942361,18.363838
min,-48.221500,8.152490,1006.370000,1.058090,0.001956,0.009557,0.009705,-12.170839,0.000000,22.011724
25%,-6.201202,14.935025,1009.250000,1.113530,0.005374,1.171170,0.238995,3.862654,0.000000,66.794079
50%,0.269070,15.581750,1011.450000,1.137810,0.007969,2.076520,0.406357,11.298311,0.000000,84.752255
75%,2.231360,16.237800,1013.950000,1.168630,0.010792,3.316100,0.625380,17.193330,219.860125,99.526525
max,35.704400,28.265000,1020.570000,1.259900,0.017989,11.932600,2.898660,32.604572,1110.706662,99.864659


# Merge data

In [5]:
df = pd.concat([df_irga72, df_irga75], axis=1)
df.describe()

,NEE_L3.1_L3.2_QCF_IRGA72,CO2_MOLAR_DENSITY_IRGA72,AIR_CP_IRGA72,AIR_DENSITY_IRGA72,VAPOR_DENSITY_IRGA72,U_IRGA72,USTAR_IRGA72,TA_T1_47_1_gfXG_IRGA72,SW_IN_T1_47_1_gfXG_IRGA72,RH_T1_47_1_IRGA72,NEE_L3.1_L3.2_QCF_IRGA75,CO2_MOLAR_DENSITY_IRGA75
count,17402.000000,27474.000000,27749.000000,27749.000000,27749.000000,27749.000000,27749.000000,28032.000000,28032.000000,28028.000000,17617.000000,31180.000000
mean,-2.866511,15.574433,1011.736249,1.141921,0.008234,2.447515,0.465091,10.725810,155.227218,80.873202,-5.472220,16.537655
std,9.000624,1.154616,2.845976,0.037616,0.003242,1.715387,0.296126,8.527703,250.942361,18.363838,10.030694,2.240562
min,-48.221500,8.152490,1006.370000,1.058090,0.001956,0.009557,0.009705,-12.170839,0.000000,22.011724,-49.996500,8.497130
25%,-6.201202,14.935025,1009.250000,1.113530,0.005374,1.171170,0.238995,3.862654,0.000000,66.794079,-10.601500,15.344950
50%,0.269070,15.581750,1011.450000,1.137810,0.007969,2.076520,0.406357,11.298311,0.000000,84.752255,-2.067410,16.026400
75%,2.231360,16.237800,1013.950000,1.168630,0.010792,3.316100,0.625380,17.193330,219.860125,99.526525,0.768966,16.810625
max,35.704400,28.265000,1020.570000,1.259900,0.017989,11.932600,2.898660,32.604572,1110.706662,99.864659,30.589900,34.866200


# Save to file

In [6]:
filename = "22_IRGA75+IRGA72_FPC_NEE-QCF11-L3.2_2016-2017"
df.to_csv(f"{filename}.csv", index=True)
save_parquet(data=df, filename=filename)

Saved file 22_IRGA75+IRGA72_FPC_NEE-QCF11-L3.2_2016-2017.parquet (0.033 seconds).


'22_IRGA75+IRGA72_FPC_NEE-QCF11-L3.2_2016-2017.parquet'

# ✅END OF NOTEBOOK

In [7]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")

Finished. 2025-12-04 16:28:05
